In [ ]:
from pathlib import Path

import shutil
import pandas as pd
from tqdm import tqdm


In [ ]:
# .env dosyasını yükle ve varsa TR_ABDOMEN_BASE değerini çevresel değişken olarak al.
import os

from dotenv import load_dotenv


load_dotenv()



In [ ]:
# Ortam değişkeni ile temel klasör tanımı.
# Eğer TR_ABDOMEN_BASE ortam değişkeni veya .env içinde tanımı varsa onu kullan, yoksa varsayılan yolu kullan.
BASE = Path(os.environ.get(
    "TR_ABDOMEN_BASE",
    r"/Users/ramazanpolat/Desktop/datasets/abdomenDataSet"
))

XLSX = BASE / "Bilgi.xlsx"
DIRS = {
    "Eğitim": BASE / "Egitim Verisi",
    "Yarışma": BASE / "Test Verisi",
}
TRAIN_DIR = DIRS["Eğitim"]
TEST_DIR = DIRS["Yarışma"]


In [ ]:

# çıktılar
# Ortam değişkeni ile temel klasör tanımı.
# Eğer TR_ABDOMEN_PROJE ortam değişkeni veya .env içinde tanımı varsa onu kullan, yoksa varsayılan yolu kullan.
ROOT = Path(os.environ.get(
    "TR_ABDOMEN_PROJE",
    r"/Users/ramazanpolat/Desktop/datasets/abdomen"
))

# çıktılar
OUT_DIR = ROOT / "outputs"
SPLIT_DIR = OUT_DIR / "splits"
CLS_DATA_DIR = OUT_DIR / "cls_data"                 # PNG / NPZ görüntüleri
DET_DATA_DIR = OUT_DIR / "det_data"                 # YOLO formatı
SEG_DATA_DIR = OUT_DIR / "seg_data"                 # nnU-Net formatı
CKPT_DIR = OUT_DIR / "checkpoints"
LOG_DIR = OUT_DIR / "logs"

In [ ]:
fold = 0

In [ ]:
out_root = Path(OUT_DIR)
fold_dir = out_root / f"fold{fold}"
for sub in ("images/train", "images/val", "labels/train", "labels/val"):
    p = fold_dir / sub
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

In [ ]:

manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
train_cases = set(pd.read_csv(SPLIT_DIR / f"fold{fold}_train.csv")["Case Number"])
val_cases = set(pd.read_csv(SPLIT_DIR / f"fold{fold}_val.csv")["Case Number"])

In [ ]:
manifest.head()

In [ ]:
include_val_negatives = True

In [ ]:



# BBox içeren kesitleri pozitif; içermeyenleri (val setinde) background olarak yaz
from src.detection import _write_dataset_yaml, _write_slice_png, _write_yolo_label


for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc=f"YOLO fold{fold}"):
    case = int(row["case"])
    if case in train_cases:
        split = "train"
    elif case in val_cases:
        split = "val"
    else:
        continue
    bboxes_raw = str(row.get("bboxes") or "")
    has_bbox = len(bboxes_raw) > 0
    if split == "train" and not has_bbox:
        # Eğitimde bbox'ı olmayan kesitleri ekleme (sinyal daha temiz)
        continue
    if not include_val_negatives and split == "val" and not has_bbox:
        continue
    try:
        img_path, h, w = _write_slice_png(row, fold_dir / "images" / split)
    except Exception as exc:
        print(f"[skip] {row['dicom_path']}: {exc}")
        continue
    label_path = fold_dir / "labels" / split / (img_path.stem + ".txt")
    _write_yolo_label(bboxes_raw, h, w, label_path)
_write_dataset_yaml(fold_dir)